# Experiment 01 — District Graph Branching

**RBLE framework:** Radon-Bifurcated Landscape Engine

This notebook validates **RBLE Eq. (6–7)**: topological Wheeler-DeWitt bifurcation closure on a district DAG. At a parent sector $\mathcal{C}_i$, an $N$-way choice event spawns child districts with mutated gravity laws and ER=EPR conductance weights $G_{ij}$ on portal edges.

**Choice entropy continuity (Eq. 4):** branch weights $p_k$ satisfy $\sum_k p_k = 1$.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "omnifold_core").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

try:
    import networkx as nx
except ImportError:
    nx = None

%matplotlib inline
plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 11})
print(f"omnifold root: {ROOT}")


## Theory: District superspace graph

Each node stores $(M_d, g_{\mu\nu}^{\text{local}}, x_d, \Lambda_d)$. A choice event implements

$$\mathcal{C}_i \to \{\mathcal{C}_{i,k}\}_{k=1}^{N}, \quad g_{\mu\nu}^{(k)} = g_{\mu\nu}^{(i)} + \delta g_k$$

with portal edges annotated by `black_hole_at` and conductance $G_{ij} = p_k$ from softmax log-odds.


In [ ]:
from omnifold_core.superspace.district_graph import DistrictGraph

graph = DistrictGraph(gravity_mutation_strength=0.05)
root_id = graph.add_district(
    mass=1.0,
    law_of_gravity=[6.674e-11, 1.0],
    coordinate=[0.0, 0.0, 0.0],
    lambda_vacuum=1.0e-120,
)
assert root_id == 0
print(f"Root district id = {root_id}, nodes = {graph.graph.number_of_nodes()}")


## Trigger a 5-choice bifurcation event


In [ ]:
NUM_CHOICES = 5
packets = graph.trigger_choice_event(root_id, NUM_CHOICES)

assert len(packets) == NUM_CHOICES
assert graph.graph.number_of_nodes() == 1 + NUM_CHOICES
assert graph.graph.number_of_edges() == NUM_CHOICES
print(f"Spawned {len(packets)} StreamPackets from parent {root_id}")


## Verify branch weights sum to unity (RBLE Eq. 4)


In [ ]:
weights = np.asarray(packets[0].branch_weights, dtype=float)
assert weights.shape == (NUM_CHOICES,)
assert np.all(weights >= 0.0)
assert np.isclose(weights.sum(), 1.0, atol=1e-12), f"weights sum = {weights.sum()}"

entropy = packets[0].shannon_entropy()
assert entropy > 0.0
print("branch_weights:", np.round(weights, 4))
print(f"Shannon entropy S = {entropy:.4f} nats")


## Per-branch conductance matches softmax weights


In [ ]:
for k, pkt in enumerate(packets):
    g_ij = graph.get_conductance(root_id, pkt.district_id)
    assert np.isclose(g_ij, weights[k], rtol=1e-9)
    assert len(pkt.phi_stream) == NUM_CHOICES

print("Conductance G_ij matches branch weights on all portal edges.")


## Gravity-law mutations increase along branch index


In [ ]:
parent_g = np.asarray(graph.graph.nodes[root_id]["law_of_gravity"], dtype=float)
child_norms = []
for pkt in packets:
    child_g = np.asarray(graph.graph.nodes[pkt.district_id]["law_of_gravity"], dtype=float)
    child_norms.append(np.linalg.norm(child_g))
    assert np.linalg.norm(child_g) >= np.linalg.norm(parent_g) - 1e-15

assert all(child_norms[i] <= child_norms[i + 1] + 1e-12 for i in range(len(child_norms) - 1))
print("Child gravity norms (monotone):", np.round(child_norms, 6))


## Visualize district DAG (NetworkX layout)


In [ ]:
pos = nx.spring_layout(graph.graph, seed=42)
node_colors = ["#2ecc71" if n == root_id else "#3498db" for n in graph.graph.nodes]
node_sizes = [900 if n == root_id else 500 for n in graph.graph.nodes]

fig, ax = plt.subplots(figsize=(10, 7))
nx.draw_networkx_nodes(graph.graph, pos, node_color=node_colors, node_size=node_sizes, ax=ax)
nx.draw_networkx_edges(graph.graph, pos, arrows=True, arrowsize=20, edge_color="#7f8c8d", ax=ax)
nx.draw_networkx_labels(graph.graph, pos, font_size=10, ax=ax)

edge_labels = {
    (u, v): f"G={graph.get_conductance(u, v):.3f}" for u, v in graph.graph.edges
}
nx.draw_networkx_edge_labels(graph.graph, pos, edge_labels=edge_labels, font_size=8, ax=ax)
ax.set_title("District DAG after 5-choice event (RBLE superspace branching)")
ax.axis("off")
plt.tight_layout()
plt.show()


## Branch weight bar chart


In [ ]:
fig, ax = plt.subplots()
ax.bar(np.arange(NUM_CHOICES), weights, color="#9b59b6", edgecolor="k")
ax.set_xlabel("Branch index k")
ax.set_ylabel("p_k")
ax.set_title("Softmax branch weights (must sum to 1)")
ax.axhline(1.0 / NUM_CHOICES, color="gray", ls="--", label="uniform")
ax.legend()
plt.tight_layout()
plt.show()


## Serialization round-trip


In [ ]:
payload = graph.to_dict()
graph2 = DistrictGraph.from_dict(payload)
assert graph2.graph.number_of_nodes() == graph.graph.number_of_nodes()
assert graph2.graph.number_of_edges() == graph.graph.number_of_edges()
print("to_dict / from_dict round-trip OK")


## Cross-check with BranchOperator softmax


In [ ]:
from omnifold_core.superspace.branch_operator import BranchOperator

bo = BranchOperator()
# BranchOperator softmax uses log-odds from choice amplitudes, not raw linspace
choice_amps = np.linspace(0.15, 0.85, NUM_CHOICES)
bo_weights = bo.compute_branch_weights(choice_amps, NUM_CHOICES)
assert np.isclose(bo_weights.sum(), 1.0)
assert np.all(bo_weights > 0.0)
print("BranchOperator weights:", np.round(bo_weights, 4))
print("DistrictGraph weights: ", np.round(weights, 4))


## Conclusions

1. `DistrictGraph.trigger_choice_event` correctly spawns $N=5$ child sectors with directed portal edges.
2. Branch weights from softmax log-odds **sum to 1**, satisfying the choice entropy continuity law (RBLE Eq. 4).
3. Conductance $G_{ij}$ on each edge equals the corresponding branch weight $p_k$.
4. Gravity-law mutations are monotone in branch index, as required by the WDW bifurcation closure (Eq. 6–7).
5. Graph serialization preserves topology for downstream conductance / master-equation experiments.
